In [1]:
import spacy
nlp = spacy.load("ro_core_news_sm")
import gensim.models.word2vec as word2vec

In [2]:
def load_embedding_model(model_path):
    """Loads the KeyedVectors model."""
    return word2vec.KeyedVectors.load_word2vec_format(model_path, binary=False)

models = {
    'word': load_embedding_model("embeddings/corola_words.300.20.vec"),
    'lemma': load_embedding_model("embeddings/corola_lemma.300.50.vec"),
    'pos': load_embedding_model("embeddings/corola_pos.300.50.vec")
}

w2c_word = models['word']
print("Loaded word embeddings for words.")

w2c_lemma = models['lemma']
print("Loaded word embeddings for lemmas.")

w2c_pos = models['pos']
print("Loaded word embeddings for POS tags.")

Loaded word embeddings for words.
Loaded word embeddings for lemmas.
Loaded word embeddings for POS tags.


Useful Mappings and function that we will use for both requirements

In [44]:
pos_mapping = {
    'NOUN': 'Nc',
    'VERB': 'Vm',
    'ADJ': 'Af',
    'ADV': 'Af',
}

In [50]:
def format_corola_pos_key(word):
    """
    Analyzes a word and returns the COROLA formatted key (e.g., Nc_animal).
    """
    doc = nlp(word)
    token = doc[0]
    prefix = pos_mapping.get(token.pos_)
    if(token.pos_ not in pos_mapping):
        print(f"Warning: POS tag '{token.pos_}' for word '{token.text}'")
    if prefix:
        return f"{prefix}_{token.lemma_}"
    return token.lemma_

Work with word embeddings (word, lemma, lemma+POS) from COROLA: for 10 inputs (nouns, verbs, adjectives), display the 10 nearest neighbors with the corresponding similarity scores.

In [30]:
test_words = ["animal", "mânca", "frumos", "casă", "alergare", "verde", "prieten", "scrie", "mare", "copac"]

In [8]:
def get_neighbors(model, key, topn=10):
    """Queries the model and returns tuples. (word, score) """
    if model and key in model:
        results = model.most_similar(key, topn=topn)
        return [(w, round(score, 4)) for w, score in results]
    return []


In [31]:
def double_print(text):
    print(text)
    f.write(text + "\n")

def print_results(res):
    if not res:
        double_print(f"   No neighbors found.")
    for i, (nb, score) in enumerate(res, 1):
        double_print(f"   {i}. {nb:<20} | {score}")



def run_neighbors(input_words, models):
    """Processes each word across all three embedding types."""
    for word in input_words:
        double_print(f"\n{'='*15} WORD: {word.upper()} {'='*15}")
        
        # 1. Word Model 
        double_print(f"\n[Model: Word]")
        res_w = get_neighbors(models['word'], word)
        print_results(res_w)
        
        # 2. Lemma Model 
        lemma = nlp(word)[0].lemma_
        double_print(f"\n[Model: Lemma] -> Query: {lemma}")
        res_l = get_neighbors(models['lemma'], lemma)
        print_results(res_l)
        
        # 3. Lemma+POS Model 
        pos_key = format_corola_pos_key(word)
        double_print(f"\n[Model: Lemma+POS] -> Query: {pos_key}")
        res_p = get_neighbors(models['pos'], pos_key)
        print_results(res_p)

with open('corola_neighbors_output.txt', 'w', encoding='utf-8') as f:  
    run_neighbors(test_words, models)


=============== WORD: ANIMAL ===============

[Model: Word]
   1. animalul             | 0.7953
   2. animalu              | 0.7555
   3. animalului           | 0.7021
   4. animale              | 0.6998
   5. animalele            | 0.6761
   6. mamifer              | 0.6705
   7. animalic             | 0.6629
   8. ierbivor             | 0.645
   9. animalule            | 0.6293
   10. animalium            | 0.6275

[Model: Lemma] -> Query: animal
   1. porcină              | 0.7006
   2. pasăre               | 0.6704
   3. bovină               | 0.6661
   4. animalier            | 0.6652
   5. ecvideu              | 0.6505
   6. rumegătoare          | 0.6502
   7. suine                | 0.6472
   8. porc                 | 0.6452
   9. rumegător            | 0.636
   10. domestic             | 0.6293

[Model: Lemma+POS] -> Query: Nc_animal
   1. Nc_pasăre            | 0.733
   2. Nc_bovină            | 0.7104
   3. Nc_porcină           | 0.6891
   4. Nc_ecvideu           | 0.6821
   

In [ ]:
analogy_map = {
    "animal": ("om", "natură"),
    "prieten": ("femeie", "bărbat"),
    "frumos": ("admirabil", "urat"),
    "casă": ("castel", "cocioabă"),
    "alerg": ("maraton", "plimbare"),
    "verde": ("pădure", "oraș"),
    "scriu": ("carte", "notă"),
    "mare": ("ocean", "pârâu"),
    "copac": ("pădure", "grădină"),
    "mănâncă": ("restaurant", "casă"),
}

In [15]:
def get_analogy(model, target, pos, neg, top_n=10):
    """Calculates vector analogy: target + pos - neg."""
    if not model: return None
    
    for k in [target, pos, neg]:
        if k not in model: return f"Missing word: {k}"
            
    results = model.most_similar(positive=[target, pos], negative=[neg], topn=top_n)
    return [(w, round(score, 4)) for w, score in results]

In [39]:

def print_results(res):
    if isinstance(res, str):
        double_print(f"   Error: {res}")
        return
    
    if not res:
        double_print(f"   No results found.")
        return
    for i, (nb, score) in enumerate(res, 1):
        
        double_print(f"   {i}. {nb:<20} | {score}")


def run_analogy(analogies, models):
    """
    Executes both the Top 10 Neighbors and the Analogies for each word 
    across all three COROLA models.
    """
    for word, (pos_w, neg_w) in analogies.items():
        double_print(f"\n{'='*20} WORD: {word.upper()} {'='*20}")
        double_print(f"Analogy Context: {word} + {pos_w} - {neg_w}")
        
        double_print(f"\n[Model: Word]")
        res_word = get_analogy(models['word'], word, pos_w, neg_w)
        double_print(f"Result for ({word} + {pos_w} - {neg_w}):")
        print_results(res_word)

        l_target = nlp(word)[0].lemma_
        l_pos = nlp(pos_w)[0].lemma_
        l_neg = nlp(neg_w)[0].lemma_
        
        double_print(f"\n[Model: Lemma] -> Keys: {l_target}, {l_pos}, {l_neg}")
        res_lemma = get_analogy(models['lemma'], l_target, l_pos, l_neg)
        double_print(f"Result for ({l_target} + {l_pos} - {l_neg}):")
        print_results(res_lemma)

        pk_target = format_corola_pos_key(word)
        pk_pos = format_corola_pos_key(pos_w)
        pk_neg = format_corola_pos_key(neg_w)
        
        double_print(f"\n[Model: Lemma+POS] -> Keys: {pk_target}, {pk_pos}, {pk_neg}")
        res_pos = get_analogy(models['pos'], pk_target, pk_pos, pk_neg)
        double_print(f"Result for ({pk_target} + {pk_pos} - {pk_neg}):")
        print_results(res_pos)

In [59]:
with open('corola_analogy_output.txt', 'w', encoding='utf-8') as f:
    run_analogy(analogy_map, models)


==================== WORD: ANIMAL ====================
Analogy Context: animal + om - natură

[Model: Word]
Result for (animal + om - natură):
   1. animalul             | 0.6335
   2. câine                | 0.6202
   3. animalu              | 0.617
   4. patruped             | 0.5696
   5. mamifer              | 0.5607
   6. omul                 | 0.5543
   7. cimpanzeu            | 0.5428
   8. animalule            | 0.5412
   9. animalic             | 0.5314
   10. lup                  | 0.5244

[Model: Lemma] -> Keys: animal, om, natură
Result for (animal + om - natură):
   1. rumegător            | 0.5042
   2. rozător              | 0.4994
   3. mamifer              | 0.4903
   4. bovină               | 0.4872
   5. porc                 | 0.4842
   6. porcină              | 0.4791
   7. patruped             | 0.4785
   8. ierbivor             | 0.4779
   9. ecvideu              | 0.4706
   10. erbivor              | 0.4686

[Model: Lemma+POS] -> Keys: Nc_animal, Nc_om, Nc_natură